##Load Bronze

#### LOAD FORMATTED CSV FILES INTO BRONZE DELTA TABLES

### Create Bronze Schema and Define base path

In [0]:
from pyspark.sql.types import *

spark.sql("""
CREATE SCHEMA IF NOT EXISTS olist_lakehouse.bronze
""")


formatted_base_path = (
    "/Volumes/olist_lakehouse/formatted/formatted_row_data/"
)


### File and Table Mapping

In [0]:
tables = [

    {
        "file": "olist_customers_dataset",
        "table": "olist_lakehouse.bronze.olist_customers_dataset"
    },

    {
        "file": "olist_geolocation_dataset",
        "table": "olist_lakehouse.bronze.olist_geolocation_dataset"
    },

    {
        "file": "olist_order_items_dataset",
        "table": "olist_lakehouse.bronze.olist_order_items_dataset"
    },

    {
        "file": "olist_order_payments_dataset",
        "table": "olist_lakehouse.bronze.olist_order_payments_dataset"
    },

    {
        "file": "olist_order_reviews_dataset",
        "table": "olist_lakehouse.bronze.olist_order_reviews_dataset"
    },

    {
        "file": "olist_orders_dataset",
        "table": "olist_lakehouse.bronze.olist_orders_dataset"
    },

    {
        "file": "olist_products_dataset",
        "table": "olist_lakehouse.bronze.olist_products_dataset"
    },

    {
        "file": "olist_sellers_dataset",
        "table": "olist_lakehouse.bronze.olist_sellers_dataset"
    },

    {
        "file": "product_category_name_translation",
        "table": "olist_lakehouse.bronze.product_category_name_translation"
    }

]

###Load CSV Files into Bronze Delta Tables

In [0]:

for item in tables:

    file_name = item["file"]
    table_name = item["table"]

    print("==================================================")
    print(f"Processing: {file_name}")
    print("==================================================")

    input_path = formatted_base_path + file_name

    # =================================================================
    # Read CSV File
    # =================================================================

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(input_path)
    )

    # =================================================================
    # Write Delta Table
    # =================================================================

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    # =================================================================
    # Validation
    # =================================================================

    row_count = (
        spark.table(table_name)
        .count()
    )

    print(f"SUCCESS: {table_name} loaded")
    print(f"Rows Loaded: {row_count}")
    print("")

print("==================================================")
print("ALL BRONZE DELTA TABLES LOADED SUCCESSFULLY")
print("==================================================")